# Proposed validation — review before it runs

**What this measures:** The target measures the share of emitted distance/size QA answers that land within the PR's own published calibration bar (rel_tol=0.05 or half-unit rounding) against a known-geometry pinhole scene, driven end-to-end through the real `vqasynth.prompts` path (`unproject_depth_map` → per-object clouds → `PromptGenerator` predicates → parsed answers) — the claim's "real-world-calibrated" mechanism, on the exact instrument the claim analysis specifies. The base branch cannot run any of the five new mechanisms (`unproject_depth_map`, `parse_extrinsics`/perspective predicates, `ground_correspondences`, S≥2 fusion), and the claim's own protocol scores the retired x10 path side-by-side inside one run (`gen.human_like_distance(10*raw)`, ~9x off vs bar >0.5), so this is a capability question measured on the CPU-observable code paths with `vggt` stubbed exactly as the repo's own tests do; the GPU VGGT/DepthPro path the PR flags as unvalidated is explicitly not measured and recorded below.

**Target metric:** `metric_answer_within_bar_rate`

Remyx wrote this test for the change in this PR. **Nothing here has been executed** — there are no outputs, and no result is being claimed.

Edit it if the measurement is wrong, then mention `@remyx validate` again and it will run what you committed. If anything is missing at run time — an import, a dependency, a device — the run reports it and repairs what it can rather than failing silently.

This notebook is the only copy of the test: `.remyx/validation.yaml` points here, and the run executes these cells top to bottom.

In [ ]:
# papermill parameters — Remyx injects variant / ref / seed here
variant = ""
ref = ""
seed = 0

## Execution context

The cells below are the test Remyx drafted, split into cells; it was written as a script, so `__file__` is set to `eval/eval_vqasynth_spatial_quality.py` (no such file is committed). This cell gives it what the command line would: its own path in `__file__`, an empty argument list so `argparse` sees no stray flags, and the papermill parameters as `REMYX_VARIANT` / `REMYX_REF` / `REMYX_SEED` for anything that wants them.

In [ ]:
import os, sys
ROOT = os.getcwd()  # the notebook runs with the repository root as its working directory
__file__ = os.path.join(ROOT, "eval/eval_vqasynth_spatial_quality.py")
sys.argv = [__file__]
for _k in ("variant", "ref", "seed"):
    _v = globals().get(_k)
    if _v not in (None, ""):
        os.environ["REMYX_" + _k.upper()] = str(_v)
print("[remyx] cwd", ROOT, "| script", __file__)

In [ ]:
"""VQASynth++ Phase 1-2 spatial-quality eval — CPU-only, capability question (no baseline arm).

Drives the REAL changed modules against the repo's own CPU fixtures, held constant:
the pinhole scene of tests/test_metric_calibration.py (focal 100 px, depth 5 m ->
0.05 m/px, two 9x9-px patches), the known extrinsics of tests/test_prompts.py
(including its vggt stub), and the two-view fixture of tests/test_correspondence_grounding.py.
Intra-run contrast: the retired x10 path via gen.human_like_distance(10*raw) — the
repo test's own idiom — must miss the bar the calibrated path meets (~9x vs <=5%).
"""
from __future__ import annotations

In [ ]:
import argparse
import importlib.util
import json
import os
import random
import re
import sys
import types

In [ ]:
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import numpy as np  # pre-existing deps import loudly: a broken env must crash
import open3d as o3d
import torch  # noqa: F401 — vqasynth.scene_fusion dependency, real install per tests/

def _install_vggt_stub():
    """Mirror tests/test_prompts.py: stub the Docker-only vggt package (only when
    absent) so the real vqasynth modules import unmodified on CPU."""
    if importlib.util.find_spec("vggt") is not None:
        return
    root = types.ModuleType("vggt"); root.__path__ = []
    models = types.ModuleType("vggt.models"); models.__path__ = []
    mv = types.ModuleType("vggt.models.vggt"); mv.VGGT = object
    utils = types.ModuleType("vggt.utils"); utils.__path__ = []
    pose = types.ModuleType("vggt.utils.pose_enc"); pose.pose_encoding_to_extri_intri = None
    geo = types.ModuleType("vggt.utils.geometry"); geo.unproject_depth_map_to_point_map = None
    sys.modules.update({"vggt": root, "vggt.models": models, "vggt.models.vggt": mv,
                        "vggt.utils": utils, "vggt.utils.pose_enc": pose,
                        "vggt.utils.geometry": geo})

In [ ]:
_install_vggt_stub()

try:  # exactly the symbols this PR adds; absence -> zeroed metrics (fails all bars)
    from vqasynth.correspondence import (
        CorrespondenceMatch,
        CorrespondenceResult,
        aggregate_correspondences,
        build_qa_pair,
        ground_correspondences,
    )
    from vqasynth.prompt_templates import (
        perspective_left_false_responses,
        perspective_left_true_responses,
    )
    from vqasynth.prompts import PromptGenerator, unproject_depth_map

    HAVE_NEW = True
except (ImportError, AttributeError):
    HAVE_NEW = False

In [ ]:
SMOKE = os.environ.get("REMYX_SMOKE") == "1"
# full run: 6 predicates x 10 rounds = 60 seeded draws;
# smoke: 1 round x 6 predicates = 6 draws — every changed code path exercised once.
N_ROUNDS = 1 if SMOKE else 10

FOCAL_PX, DEPTH_M, WIDTH_PX, HEIGHT_PX = 100.0, 5.0, 161, 121
GT_SIZE_M, GT_VERTICAL_M, GT_HORIZONTAL_M = 0.4, 0.4, 0.8
_ANSWER_RE = re.compile(r"([0-9]+(?:\.[0-9]+)?)\s*(meters|centimeters|inches|feet)")
_UNITS = {"meters": 1.0, "centimeters": 0.01, "inches": 0.0254, "feet": 0.3048}
# Exact regex vqasynth.localize uses — the round-trip contract for <point> tags.
_LOCALIZE_POINT_RE = re.compile(
    r'<point\s+x="\s*([0-9]+(?:\.[0-9]+)?)"\s+y="\s*([0-9]+(?:\.[0-9]+)?)"\s+alt="([^"]+)">'
)

In [ ]:
def _grid_meters():
    us, vs = np.meshgrid(np.arange(WIDTH_PX), np.arange(HEIGHT_PX))
    cx, cy = (WIDTH_PX - 1) / 2.0, (HEIGHT_PX - 1) / 2.0
    return (us - cx) / FOCAL_PX * DEPTH_M, (vs - cy) / FOCAL_PX * DEPTH_M

def _patch_masks():
    x_m, y_m = _grid_meters()
    eps = 1e-9
    return (
        (x_m >= -0.6 - eps) & (x_m <= -0.2 + eps) & (y_m >= -eps) & (y_m <= 0.4 + eps),
        (x_m >= 0.2 - eps) & (x_m <= 0.6 + eps) & (y_m >= -0.4 - eps) & (y_m <= eps),
    )

In [ ]:
def _cloud(points):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(np.asarray(points, dtype=np.float64))
    return pcd

def _parse_meters(text):
    m = _ANSWER_RE.search(text)
    if m is None:
        return 0.0, "meters", False
    return float(m.group(1)) * _UNITS[m.group(2)], m.group(2), True

In [ ]:
def _emit(metrics):
    print(json.dumps({k: round(float(v), 6) for k, v in metrics.items()}))

def main():
    if not HAVE_NEW:  # PR symbols absent: express failure as values, not a crash-hide
        _emit({"metric_answer_within_bar_rate": 0.0, "localize_roundtrip_rate": 0.0,
               "perspective_flip_accuracy": 0.0, "metric_answer_rel_error": 1.0,
               "retired_x10_rel_error": 0.0, "grounded_label_coverage": 0.0,
               "inlier_ratio_mean": 0.0, "unusable_extrinsics_accept_rate": 1.0})
        return

    random.seed(0)  # template/unit draws, mirroring the repo's own test seeding
    gen = PromptGenerator()

    # ── target: calibrated distance/size answers vs known ground truth ──
    depth_map = np.full((HEIGHT_PX, WIDTH_PX), DEPTH_M, dtype=np.float64)
    points = unproject_depth_map(depth_map, FOCAL_PX)
    mask_a, mask_b = _patch_masks()
    cloud_a, cloud_b = _cloud(points[mask_a.ravel()]), _cloud(points[mask_b.ravel()])
    x_m, y_m = _grid_meters()
    gt_pairwise = float(np.mean(np.hypot(0.2 - x_m[mask_a], y_m[mask_a])))  # in [0.632, 0.658]
    raw = float(np.asarray(cloud_a.compute_point_cloud_distance(cloud_b)).mean())
    A, B = ("red cube", cloud_a), ("blue ball", cloud_b)
    draws = [
        (lambda: gen.vertical_distance_data(A, B), GT_VERTICAL_M),
        (lambda: gen.horizontal_distance_data(A, B), GT_HORIZONTAL_M),
        (lambda: gen.width_data(A), GT_SIZE_M),
        (lambda: gen.height_data(A), GT_SIZE_M),
        (lambda: gen.height_data(B), GT_SIZE_M),
        (lambda: gen.human_like_distance(gen._metric(raw)), gt_pairwise),
    ]
    # Bar (PR): |answer - GT| <= max(0.05*GT, half the emitted unit's rounding).
    # Unprojection is exact on this scene, so only unit rounding moves answers and
    # the half-unit branch absorbs it -> expected 1.0; an x10 leak (~9x) cannot pass.
    hits, rel_errs = 0, []
    for _ in range(N_ROUNDS):
        for produce, gt in draws:
            got_m, unit, ok = _parse_meters(produce())
            hits += ok and abs(got_m - gt) <= max(0.05 * gt, 0.5 * _UNITS[unit])
            rel_errs.append(abs(got_m - gt) / gt if ok else 1.0)
    within_rate = hits / float(N_ROUNDS * len(draws))
    rel_error = float(np.mean(rel_errs))
    got10, _, ok10 = _parse_meters(gen.human_like_distance(10 * raw))  # retired x10 idiom
    x10_rel = abs(got10 - gt_pairwise) / gt_pairwise if ok10 else 1.0

    # ── perspective flips: world-left object seen right from a 180-degree-yaw camera ──
    identity = np.hstack([np.eye(3), np.zeros((3, 1))])
    yaw180 = np.hstack([np.diag([-1.0, 1.0, -1.0]), np.zeros((3, 1))])
    extrinsics = np.stack([identity, yaw180])
    P = ("red mug", _cloud(np.array([[-1.0, 0.0, 5.0]])))
    Q = ("blue cup", _cloud(np.array([[1.0, 0.0, 5.0]])))

    def _in_bank(answer, bank, view):
        return answer in [t.replace("[A]", "red mug").replace("[B]", "blue cup")
                          .replace("[V]", view) for t in bank]

    _, cam1 = gen.perspective_left_predicate(P, Q, 0, extrinsics).split(" Answer: ")
    _, cam2 = gen.perspective_left_predicate(P, Q, 1, extrinsics).split(" Answer: ")
    flip_acc = (_in_bank(cam1, perspective_left_true_responses, "camera 1")
                + _in_bank(cam2, perspective_left_false_responses, "camera 2")) / 2.0

    # ── gate: unusable extrinsics (None / <2 frames / NaN / rotation-only) ──
    unusable = [None, identity, [identity], np.full((2, 3, 4), np.nan),
                np.eye(3)[None], [{"rotation": np.eye(3)}]]
    leak_rate = sum(gen.parse_extrinsics(u) is not None for u in unusable) / len(unusable)

    # ── correspondence grounding, inlier-ratio column, localize round-trip ──
    def _rect(h, w, y0, y1, x0, x1):
        return [[y0 <= y < y1 and x0 <= x < x1 for x in range(w)] for y in range(h)]

    size = (10, 10)
    masks_a = [_rect(10, 10, 0, 5, 0, 5), _rect(10, 10, 5, 10, 5, 10)]
    masks_b = [_rect(10, 10, 0, 5, 5, 10), _rect(10, 10, 5, 10, 5, 10)]
    captions = ["wooden chair", "desk"]

    def _result(matches, inliers, raws):
        return CorrespondenceResult(matches=matches, view_a_size=size,
                                    view_b_size=size, inlier_count=inliers,
                                    raw_match_count=raws)

    fixture = _result([CorrespondenceMatch(pt_a=(2, 2), pt_b=(7, 2)),
                       CorrespondenceMatch(pt_a=(3, 8), pt_b=(7, 7)),
                       CorrespondenceMatch(pt_a=(8, 0), pt_b=(0, 9))], 3, 6)
    grounded = ground_correspondences(fixture, masks_a=masks_a, captions_a=captions,
                                      masks_b=masks_b, captions_b=captions)
    coverage = (sum(m.caption != "feature" for m in grounded.matches)
                / max(len(grounded.matches), 1))  # fixture bar: 2 of 3 grounded
    tags = []
    for m in grounded.matches:  # grounded AND un-grounded matches both emit tags
        q, a = build_qa_pair(m, size, size)
        tags += re.findall(r"<point[^>]*>", q) + re.findall(r"<point[^>]*>", a)
    roundtrip = (sum(_LOCALIZE_POINT_RE.search(t) is not None for t in tags)
                 / max(len(tags), 1))
    row = aggregate_correspondences([
        _result([CorrespondenceMatch(pt_a=(1, 1), pt_b=(2, 2))], 8, 10),  # ratio 0.8
        _result([CorrespondenceMatch(pt_a=(3, 3), pt_b=(4, 4))], 5, 10),  # ratio 0.5
    ])
    inlier_mean = float(row["inlier_ratio"])  # surfaced column: mean(0.8, 0.5) = 0.65

    _emit({
        "metric_answer_within_bar_rate": within_rate,
        "localize_roundtrip_rate": roundtrip,
        "perspective_flip_accuracy": flip_acc,
        "metric_answer_rel_error": rel_error,
        "retired_x10_rel_error": x10_rel,
        "grounded_label_coverage": coverage,
        "inlier_ratio_mean": inlier_mean,
        "unusable_extrinsics_accept_rate": leak_rate,
    })

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    for flag in ("--variant", "--ref", "--seed"):
        parser.add_argument(flag, default=None)  # accepted and ignored per contract
    parser.parse_known_args()
    main()

## The criteria this is judged against

From `.remyx/validation.yaml` — thresholds live here, not in the test, so a failing measurement reports rather than crashes.

```yaml
question:
  kind: capability
  ask: "Does VQASynth++ Phase 1-2 deliver real-world-calibrated, viewpoint-aware spatial data (calibrated distance/size, perspective flips, grounded correspondence) with the single-image contracts preserved?"
benchmarks:
  - name: vqasynth-pp-spatial-quality
    suite: "eval/eval_vqasynth_spatial_quality.py"
    scorer: metric_answer_within_bar_rate
    baseline: none
    policy:
      guardrail_veto: true
    metrics:
      - name: metric_answer_within_bar_rate
        direction: max
        threshold: 1.0
        role: target
        bar: goal
        reads_as: "Share of emitted distance/size answers matching known ground-truth meters within the PR bar (rel_tol=0.05 or half-unit rounding)"
        # Threshold derivation: the calibrated path is exact pinhole unprojection on a
        # synthetic scene (focal 100px, depth 5m -> 0.05 m/px), so answers move only by
        # the humanizer's unit rounding, which the bar's half-unit branch absorbs
        # (worst case 0.4m->whole feet: |0.3048-0.4|=0.095 <= max(0.02, 0.1524)); only
        # genuine miscalibration (e.g. an x10 leak) misses, so 1.0 over 60 seeded draws.
      - name: localize_roundtrip_rate
        direction: max
        threshold: 1.0
        role: guardrail
        bar: floor
        reads_as: "Share of emitted <point> tags (grounded and un-grounded) that parse through the localize stage's exact regex"
      - name: perspective_flip_accuracy
        direction: max
        threshold: 1.0
        role: diagnostic
        reads_as: "Share of egocentric left/right answers that flip correctly across the 180-degree-yaw second camera (fixture bar 2/2)"
      - name: metric_answer_rel_error
        direction: min
        threshold: 0.05
        role: diagnostic
        reads_as: "Mean relative miss of calibrated answers vs ground truth; typical residual is unit rounding only"
      - name: retired_x10_rel_error
        direction: max
        threshold: 0.5
        role: diagnostic
        reads_as: "Relative error of the retired x10 path on the same scene (paper bar >0.5, expected ~9x) - the intra-run contrast"
      - name: grounded_label_coverage
        direction: max
        threshold: 0.66
        role: diagnostic
        reads_as: "Share of fixture correspondence matches grounded to object labels instead of the anonymous 'feature'"
      - name: inlier_ratio_mean
        direction: max
        threshold: 0.65
        role: diagnostic
        reads_as: "Mean surfaced inlier_ratio over fixture results (0.8 and 0.5 -> 0.65; 0/0 guarded)"
      - name: unusable_extrinsics_accept_rate
        direction: min
        threshold: 0.0
        role: diagnostic
        reads_as: "Share of unusable extrinsics (None, <2 frames, NaN, rotation-only) that parse_extrinsics wrongly accepts -> perspective leak into S=1 data"
compute:
  tier: cpu
held_constant:
  - "Synthetic pinhole scene held exactly as tests/test_metric_calibration.py: focal 100 px, depth 5 m, 161x121 grid, two 9x9-px patches (GT size 0.4 m, vertical 0.4 m, horizontal 0.8 m, pairwise mean-NN in [0.632, 0.658] m)"
  - "random.seed(0) for template/unit draws, mirroring the repo's own test"
  - "vggt stub installed before importing vqasynth exactly as tests/test_prompts.py does, only when the real package is absent"
  - "OpenCV convention (x right, y down, z forward) and the localize <point> regex held as the round-trip contract"
  - "No GPU, no model downloads, no judge calls: repo CPU-test philosophy per tests/test_metric_calibration.py and tests/test_correspondence.py"
avoid:
  - "pytest pass counts as the quality instrument (repo rule: pytest proves code-works only)"
  - "Guessing signatures of vqasynth.scene_fusion internals or calibrate_metric_scale/depth_map_to_meters - not exercised because their definitions are not visible in the provided source"
  - "Real VGGT/DepthPro model calls (vggt is not in requirements.txt and is Docker-only; stubbed per the repo's own tests)"
  - "Reimplementing the retired x10 path other than via gen.human_like_distance(10*raw), the idiom the repo's own retired-path test uses"
  - "Wall-clock timing on shared CPU"
provenance:
  user_guidance: "@remyx-ai validate - requester asked only to run validation; no dataset, loader, model, or metric override named, so scope and metrics follow the PR diff and the repo's own CPU fixtures unchanged"
  metric_answer_within_bar_rate: "claim_analysis:paper_metrics (rel_tol=0.05 or half-unit rounding bar)"
  metric_answer_rel_error: "claim_analysis:paper_metrics"
  retired_x10_rel_error: "claim_analysis:paper_metrics + arithmetic (x10 ~9x off, must exceed 0.5)"
  perspective_flip_accuracy: "claim_analysis:arithmetic (yaw-180 flips left/right exactly; fixture 2/2)"
  grounded_label_coverage: "claim_analysis:paper_metrics (2 of 3 fixture matches grounded)"
  inlier_ratio_mean: "claim_analysis:paper_metrics (0.8, 0.5 -> 0.65; 0/0 -> 0.0)"
  unusable_extrinsics_accept_rate: "claim_analysis:mechanism (parse_extrinsics returns None for <2 frames, NaN, rotation-only)"
  localize_roundtrip_rate: "repo_test:tests/test_correspondence.py LOCALIZE_POINT_RE contract"
  held_constant: "repo_test:tests/test_metric_calibration.py scene geometry"
  suite: "synthesized"
  compute: "inferred"
report:
  headline: "Metric calibration and perspective logic hold at their published bars on known geometry while the retired x10 path misses by ~9x; the real VGGT/DepthPro GPU path remains unmeasured"
  findings:
    - "{metric_answer_within_bar_rate:%} of emitted distance/size answers land within the PR's rel_tol=0.05-or-half-unit bar against known ground truth (mean relative miss {metric_answer_rel_error}), while the retired x10 path misses by {retired_x10_rel_error} relative"
    - "{perspective_flip_accuracy:%} of egocentric left/right answers flip correctly across the 180-degree-yaw second camera, and {unusable_extrinsics_accept_rate:%} of unusable extrinsics leak into perspective QA"
    - "{grounded_label_coverage:%} of fixture correspondence matches ground to object labels instead of 'feature', with surfaced inlier_ratio averaging {inlier_ratio_mean}"
    - "{localize_roundtrip_rate:%} of emitted <point> tags still parse through the localize regex, keeping the single-image pipeline contract intact"
  establishes:
    - "Calibrated distance/size QA is metrically correct on known-geometry scenes, resting on {metric_answer_within_bar_rate} and the {retired_x10_rel_error} contrast"
    - "Perspective-taking answers flip with viewpoint as claimed, resting on {perspective_flip_accuracy} and the {unusable_extrinsics_accept_rate} gate"
    - "Correspondence QA is object-grounded with an inlier-ratio column, resting on {grounded_label_coverage} and {inlier_ratio_mean}"
  does_not_establish:
    - "Quality of real VGGT depth/extrinsics or the DepthPro metric anchor (calibrate_metric_scale) on real imagery - no GPU model ran"
    - "Byte-for-byte S=1 scene_fusion parity vs main - no baseline arm and no VGGT-format S=1 outputs were diffed"
    - "Any downstream VLM training gain from the new QA families"
  not_measured:
    - "Multi-view fusion registration error on real clips (S>=2 with predicted extrinsics)"
    - "Per-object depth confidence (compute_object_depth_confidence) - its signature is not visible in the provided source"
    - "Judge-scored data quality via experiments/prometheus_space_judge (R2: no canonical rows; runner config schema and emitted fields not visible in context)"
  caveat: "The PR's own evidence is 40 CPU pytest passes; this eval upgrades the same synthetic-geometry mechanisms to measured values but still cannot touch the GPU-gated VGGT/DepthPro path the PR itself flags as unvalidated."
  next: "Run the prometheus_space_judge harness on a >=2-frame clip sample on GPU with a new experiments/ config directory to measure real-VGGT fusion, anchor calibration, and judge-scored spatial QA quality."
loop:
  max_iterations: 8
  fix_code: true
```